# V2 Phase 7 — Colab GPU smoke (Qwen3-8B)

**Before running:** Runtime → Change runtime type → **GPU**.

## Setup instructions

Your GitHub repo has **two top-level folders**:

```text
repo/
├── .cursor/rules/     ← Cursor rules (ignore in Colab)
└── V2/                ← MSc project code — Colab uses THIS folder
```

1. **Push latest changes** to branch `cursor/empty-v2-workspace`
2. Open this notebook with **GPU** runtime
3. Run all cells — cell 1 clones the repo, then **`cd` into `V2/`**

No need to upload anything to Google Drive for source code.

**Outputs:** `V2/results/config/phase7_smoke_test.json` (after smoke run)

## 1. Clone GitHub repo and enter V2

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/syedsafiullah777/CAPSTONE--RAG-WITH-UNCERTAINITY-QUANTIFICATION-.git'
BRANCH = 'cursor/empty-v2-workspace'  # only branch on this repo
CLONE_DIR = Path('/content/capstone-rag')

if CLONE_DIR.exists():
    !rm -rf {CLONE_DIR}

print('Cloning branch:', BRANCH)
result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CLONE_DIR)],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'git clone failed. Push V2/ to GitHub on branch {BRANCH!r} first.')

print('Repo root contents:', [p.name for p in CLONE_DIR.iterdir()])
# GitHub layout: .cursor/ + V2/ — we work inside V2/
V2_ROOT = CLONE_DIR / 'V2'
if not V2_ROOT.is_dir():
    raise FileNotFoundError(
        f'Missing V2/ folder at {V2_ROOT}. '
        'Your repo should have .cursor/rules/ and V2/ at the top level.'
    )
if not (V2_ROOT / 'scripts' / 'smoke_generate.py').is_file():
    raise FileNotFoundError(f'Invalid V2 root: {V2_ROOT}')

os.chdir(V2_ROOT)
sys.path.insert(0, str(V2_ROOT))
print('OK — working in V2_ROOT:', V2_ROOT)
!git -C {CLONE_DIR} log -1 --oneline

## 2. Install dependencies

In [ ]:
!pip -q install -r requirements.txt
!pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

## 3. Run smoke test (llama_cpp on Colab GPU)

In [ ]:
!PYTHONPATH=. python scripts/smoke_generate.py --backend llama_cpp --notebook notebooks/colab_phase7_smoke.ipynb

In [ ]:
# Fallback only if llama_cpp fails:
# !PYTHONPATH=. python scripts/smoke_generate.py --backend transformers --notebook notebooks/colab_phase7_smoke.ipynb

## 4. Check results

In [ ]:
import json
from pathlib import Path

fp = Path('results/config/phase7_runtime_fingerprint.json')
smoke = Path('results/config/phase7_smoke_test.json')
print('fingerprint:', fp.is_file())
print('smoke_test:', smoke.is_file())
if smoke.is_file():
    data = json.loads(smoke.read_text())
    print('status:', data.get('status'))
    print('actual:', repr(data.get('actual')))

## 5. (Optional) Copy results to Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
dest = Path('/content/drive/MyDrive/MSc-RAG/configs/phase7')
dest.mkdir(parents=True, exist_ok=True)
for name in ('phase7_runtime_fingerprint.json', 'phase7_smoke_test.json'):
    src = Path('results/config') / name
    if src.is_file():
        shutil.copy2(src, dest / name)
        print('copied', name)